# TP2 — N-grams, Context Window and Bag of Words

In this TP we are going to implement:
1. A text preprocessing pipeline (manual, no magic functions)
2. N-gram extraction and BoW vectorization
3. KMeans clustering and comparison between 1-gram and 2-gram results
4. Context Window vectorization with start/end padding

---

In [44]:
# ============================================================
# PART 0 — Preprocessing helpers
# ============================================================


PUNCT = set(".,!?;:\"'()-[]{}\\/@#$%^&*_+=<>~`|")

CONTRACTIONS = {
    "won't": "will not",
    "can't": "cannot",
    "don't": "do not",
    "doesn't": "does not",
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",
    "i'm": "i am",
    "i've": "i have",
    "i'll": "i will",
    "i'd": "i would",
    "it's": "it is",
    "that's": "that is",
    "there's": "there is",
    "they're": "they are",
    "we're": "we are",
    "you're": "you are",
}


STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "in", "on", "at",
    "to", "of", "for", "with", "is", "are", "was", "were",
    "be", "been", "being", "have", "has", "had", "do", "does",
    "did", "will", "would", "could", "should", "may", "might",
    "it", "its", "this", "that", "these", "those",
    "i", "me", "my", "we", "our", "you", "your", "he", "she",
    "they", "their", "them", "not", "so", "if", "as", "from",
}


def expand_contractions(text):
    """Replace contractions with full forms."""
    words = text.split()
    expanded = []
    for w in words:
        lower_w = w.lower()
        if lower_w in CONTRACTIONS:
            expanded.append(CONTRACTIONS[lower_w])
        else:
            expanded.append(w)
    return " ".join(expanded)


def remove_punctuation(text):
    """Strip punctuation character by character."""
    clean = ""
    for ch in text:
        if ch not in PUNCT:
            clean += ch
        else:
            clean += " "
    return clean


def tokenize(text):
    """Split text into tokens (words), manually."""

    tokens = []
    current_word = ""
    for ch in text:
        if ch == " " or ch == "\t" or ch == "\n":
            if current_word != "":
                tokens.append(current_word)
                current_word = ""
        else:
            current_word += ch
    if current_word != "":
        tokens.append(current_word)
    return tokens


def preprocess_text(text, remove_stops=False):
    """
    Full preprocessing pipeline:
    lowercase → expand contractions → remove punctuation → tokenize → (optional) remove stopwords
    Returns a list of tokens.
    """
    text = text.lower()
    text = expand_contractions(text)
    text = remove_punctuation(text)
    tokens = tokenize(text)
    if remove_stops:
        tokens = [t for t in tokens if t not in STOPWORDS]

    return tokens


test = "The gold medal price is high effort"
print("Original:", test)
print("Preprocessed:", preprocess_text(test))

Original: The gold medal price is high effort
Preprocessed: ['the', 'gold', 'medal', 'price', 'is', 'high', 'effort']


In [45]:
def extract_ngrams(tokens, n):
    ngrams = []
    for i in range(len(tokens) - n + 1):
        group = tokens[i : i + n]
        ngrams.append(" ".join(group))
    return ngrams


sample_tokens = ["i", "love", "cats"]
print("Unigrams (n=1):", extract_ngrams(sample_tokens, 1))
print("Bigrams (n=2):", extract_ngrams(sample_tokens, 2))
print("Trigrams (n=3):", extract_ngrams(sample_tokens, 3))

Unigrams (n=1): ['i', 'love', 'cats']
Bigrams (n=2): ['i love', 'love cats']
Trigrams (n=3): ['i love cats']


In [46]:
def build_vocabulary(all_docs, n):
    all_ngrams = set()

    for doc in all_docs:
        tokens = preprocess_text(doc)
        ngrams = extract_ngrams(tokens, n)
        for ng in ngrams:
            all_ngrams.add(ng)

    vocab = sorted(list(all_ngrams))

    vocab_index = {}
    for idx, ng in enumerate(vocab):
        vocab_index[ng] = idx

    return vocab, vocab_index


doc1 = "The gold medal price is high effort"
doc2 = "Winning a gold medal needs a high jump"
doc3 = "Market for a gold medal is a trade of sweat"
doc4 = "The athlete will trade all for a gold medal"

doc5 = "The gold bars price is high today"
doc6 = "Investing in gold bars needs a high rate"
doc7 = "Market for gold bars is a trade of money"
doc8 = "The bank will trade all for gold bars"

all_docs = [doc1, doc2, doc3, doc4, doc5, doc6, doc7, doc8]

vocab_1, vocab_idx_1 = build_vocabulary(all_docs, n=1)
print(f"=== 1-gram Vocabulary ({len(vocab_1)} tokens) ===")
for i, v in enumerate(vocab_1):
    print(f"  [{i}] {v}")

=== 1-gram Vocabulary (26 tokens) ===
  [0] a
  [1] all
  [2] athlete
  [3] bank
  [4] bars
  [5] effort
  [6] for
  [7] gold
  [8] high
  [9] in
  [10] investing
  [11] is
  [12] jump
  [13] market
  [14] medal
  [15] money
  [16] needs
  [17] of
  [18] price
  [19] rate
  [20] sweat
  [21] the
  [22] today
  [23] trade
  [24] will
  [25] winning


In [47]:
vocab_2, vocab_idx_2 = build_vocabulary(all_docs, n=2)
print(f"=== 2-gram Vocabulary ({len(vocab_2)} bigrams) ===")
for i, v in enumerate(vocab_2):
    print(f"  [{i}] {v}")

=== 2-gram Vocabulary (36 bigrams) ===
  [0] a gold
  [1] a high
  [2] a trade
  [3] all for
  [4] athlete will
  [5] bank will
  [6] bars is
  [7] bars needs
  [8] bars price
  [9] for a
  [10] for gold
  [11] gold bars
  [12] gold medal
  [13] high effort
  [14] high jump
  [15] high rate
  [16] high today
  [17] in gold
  [18] investing in
  [19] is a
  [20] is high
  [21] market for
  [22] medal is
  [23] medal needs
  [24] medal price
  [25] needs a
  [26] of money
  [27] of sweat
  [28] price is
  [29] the athlete
  [30] the bank
  [31] the gold
  [32] trade all
  [33] trade of
  [34] will trade
  [35] winning a


In [48]:
def vectorize_doc_boolean(doc, vocab_index, n):
    tokens = preprocess_text(doc)
    ngrams = extract_ngrams(tokens, n)

    doc_ngrams_set = set(ngrams)

    vector = [0] * len(vocab_index)
    for ng, idx in vocab_index.items():
        if ng in doc_ngrams_set:
            vector[idx] = 1

    return vector


def vectorize_doc_count(doc, vocab_index, n):
    tokens = preprocess_text(doc)
    ngrams = extract_ngrams(tokens, n)

    counts = {}
    for ng in ngrams:
        if ng in counts:
            counts[ng] += 1
        else:
            counts[ng] = 1

    vector = [0] * len(vocab_index)
    for ng, idx in vocab_index.items():
        if ng in counts:
            vector[idx] = counts[ng]

    return vector


def vectorize_all(docs, n, mode="boolean"):
    vocab, vocab_index = build_vocabulary(docs, n)

    matrix = []
    for doc in docs:
        if mode == "boolean":
            vec = vectorize_doc_boolean(doc, vocab_index, n)
        else:
            vec = vectorize_doc_count(doc, vocab_index, n)
        matrix.append(vec)

    return matrix, vocab


matrix_1, vocab_1 = vectorize_all(all_docs, n=1, mode="boolean")

print("=== 1-gram Boolean BoW Matrix ===")
print(f"Matrix dimensions: {len(matrix_1)} docs × {len(vocab_1)} features")
print()

header = " " * 6 + "  ".join([v[:8].ljust(8) for v in vocab_1])
print(header)
print("-" * len(header))

for i, row in enumerate(matrix_1):
    label = f"doc{i+1}: "
    print(label + "  ".join([str(v).ljust(8) for v in row]))

=== 1-gram Boolean BoW Matrix ===
Matrix dimensions: 8 docs × 26 features

      a         all       athlete   bank      bars      effort    for       gold      high      in        investin  is        jump      market    medal     money     needs     of        price     rate      sweat     the       today     trade     will      winning 
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
doc1: 0         0         0         0         0         1         0         1         1         0         0         1         0         0         1         0         0         0         1         0         0         1         0         0         0         0       
doc2: 1         0         0         0         0         0         0         1         1         0         0         0         1   

In [49]:
matrix_2, vocab_2 = vectorize_all(all_docs, n=2, mode="boolean")

print("=== 2-gram Boolean BoW Matrix ===")
print(f"Matrix dimensions: {len(matrix_2)} docs × {len(vocab_2)} features")
print()
print("Vocab (2-grams):")
for i, v in enumerate(vocab_2):
    print(f"  [{i:2d}] {v}")
print()

for i, row in enumerate(matrix_2):
    label = f"doc{i+1}: "
    print(label + str(row))

=== 2-gram Boolean BoW Matrix ===
Matrix dimensions: 8 docs × 36 features

Vocab (2-grams):
  [ 0] a gold
  [ 1] a high
  [ 2] a trade
  [ 3] all for
  [ 4] athlete will
  [ 5] bank will
  [ 6] bars is
  [ 7] bars needs
  [ 8] bars price
  [ 9] for a
  [10] for gold
  [11] gold bars
  [12] gold medal
  [13] high effort
  [14] high jump
  [15] high rate
  [16] high today
  [17] in gold
  [18] investing in
  [19] is a
  [20] is high
  [21] market for
  [22] medal is
  [23] medal needs
  [24] medal price
  [25] needs a
  [26] of money
  [27] of sweat
  [28] price is
  [29] the athlete
  [30] the bank
  [31] the gold
  [32] trade all
  [33] trade of
  [34] will trade
  [35] winning a

doc1: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0]
doc2: [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
doc3: [1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0

In [50]:
import math
import random


def euclidean_distance(a, b):
    total = 0.0
    for i in range(len(a)):
        diff = a[i] - b[i]
        total += diff * diff
    return math.sqrt(total)


def compute_centroid(points):
    if len(points) == 0:
        return None

    n_dims = len(points[0])
    centroid = [0.0] * n_dims

    for point in points:
        for d in range(n_dims):
            centroid[d] += point[d]

    for d in range(n_dims):
        centroid[d] /= len(points)

    return centroid


def my_kmeans(data, k=2, max_iters=100, seed=42):
    random.seed(seed)
    n = len(data)

    indices = random.sample(range(n), k)
    centroids = [data[i][:] for i in indices]

    labels = [0] * n

    for iteration in range(max_iters):

        new_labels = []
        for point in data:
            best_cluster = 0
            best_dist = float("inf")
            for c_idx, centroid in enumerate(centroids):
                dist = euclidean_distance(point, centroid)
                if dist < best_dist:
                    best_dist = dist
                    best_cluster = c_idx
            new_labels.append(best_cluster)

        if new_labels == labels:
            print(f"  KMeans converged after {iteration} iterations.")
            break

        labels = new_labels

        for c_idx in range(k):
            cluster_points = [data[i] for i in range(n) if labels[i] == c_idx]
            if len(cluster_points) > 0:
                centroids[c_idx] = compute_centroid(cluster_points)

    return labels


print("=" * 50)
print("KMeans on 1-gram BoW vectors (manual):")
labels_1 = my_kmeans(matrix_1, k=2, seed=42)
print(f"  Cluster labels: {labels_1}")
print(f"  docs 1-4 (sports):  {labels_1[:4]}")
print(f"  docs 5-8 (finance): {labels_1[4:]}")

print()

print("=" * 50)
print("KMeans on 2-gram BoW vectors (manual):")
labels_2 = my_kmeans(matrix_2, k=2, seed=42)
print(f"  Cluster labels: {labels_2}")
print(f"  docs 1-4 (sports):  {labels_2[:4]}")
print(f"  docs 5-8 (finance): {labels_2[4:]}")

KMeans on 1-gram BoW vectors (manual):
  KMeans converged after 1 iterations.
  Cluster labels: [1, 0, 0, 0, 1, 0, 0, 1]
  docs 1-4 (sports):  [1, 0, 0, 0]
  docs 5-8 (finance): [1, 0, 0, 1]

KMeans on 2-gram BoW vectors (manual):
  KMeans converged after 1 iterations.
  Cluster labels: [1, 0, 0, 0, 1, 0, 1, 1]
  docs 1-4 (sports):  [1, 0, 0, 0]
  docs 5-8 (finance): [1, 0, 1, 1]


In [51]:
import numpy as np
from sklearn.cluster import KMeans

X1 = np.array(matrix_1)
X2 = np.array(matrix_2)

km1 = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X1)
km2 = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X2)

print("sklearn results:")
print(f"  1-gram clusters: {list(km1.labels_)}")
print(f"  2-gram clusters: {list(km2.labels_)}")

print()
print("Our manual results:")
print(f"  1-gram clusters: {labels_1}")
print(f"  2-gram clusters: {labels_2}")

sklearn results:
  1-gram clusters: [np.int32(1), np.int32(1), np.int32(0), np.int32(0), np.int32(1), np.int32(1), np.int32(0), np.int32(0)]
  2-gram clusters: [np.int32(0), np.int32(1), np.int32(0), np.int32(0), np.int32(0), np.int32(1), np.int32(0), np.int32(0)]

Our manual results:
  1-gram clusters: [1, 0, 0, 0, 1, 0, 0, 1]
  2-gram clusters: [1, 0, 0, 0, 1, 0, 1, 1]


In [52]:
true_labels = [0, 0, 0, 0, 1, 1, 1, 1]


def clustering_accuracy(predicted, true):
    n = len(true)

    correct_a1 = sum(1 for p, t in zip(predicted, true) if p == t)
    acc_a1 = correct_a1 / n

    flipped = [1 - p for p in predicted]
    correct_a2 = sum(1 for p, t in zip(flipped, true) if p == t)
    acc_a2 = correct_a2 / n

    best_acc = max(acc_a1, acc_a2)
    best_alignment = "normal" if acc_a1 >= acc_a2 else "flipped"

    return best_acc, best_alignment


def show_cluster_report(labels, true, name):
    acc, alignment = clustering_accuracy(labels, true)

    print(f"\n{'='*45}")
    print(f"  Results for: {name}")
    print(f"{'='*45}")
    print(f"  Predicted labels : {labels}")
    print(f"  True labels      : {true}")
    print(f"  Label alignment  : {alignment}")
    print(f"  Accuracy         : {acc * 100:.1f}%")

    working_labels = labels if alignment == "normal" else [1 - l for l in labels]
    for i in range(len(true)):
        status = "✓" if working_labels[i] == true[i] else "✗"
        group = "Sports " if true[i] == 0 else "Finance"
        print(f"  doc{i+1} ({group}) → cluster {labels[i]}  {status}")


show_cluster_report(labels_1, true_labels, "Manual KMeans — 1-gram")
show_cluster_report(labels_2, true_labels, "Manual KMeans — 2-gram")

show_cluster_report(list(km1.labels_), true_labels, "sklearn KMeans — 1-gram")
show_cluster_report(list(km2.labels_), true_labels, "sklearn KMeans — 2-gram")


  Results for: Manual KMeans — 1-gram
  Predicted labels : [1, 0, 0, 0, 1, 0, 0, 1]
  True labels      : [0, 0, 0, 0, 1, 1, 1, 1]
  Label alignment  : normal
  Accuracy         : 62.5%
  doc1 (Sports ) → cluster 1  ✗
  doc2 (Sports ) → cluster 0  ✓
  doc3 (Sports ) → cluster 0  ✓
  doc4 (Sports ) → cluster 0  ✓
  doc5 (Finance) → cluster 1  ✓
  doc6 (Finance) → cluster 0  ✗
  doc7 (Finance) → cluster 0  ✗
  doc8 (Finance) → cluster 1  ✓

  Results for: Manual KMeans — 2-gram
  Predicted labels : [1, 0, 0, 0, 1, 0, 1, 1]
  True labels      : [0, 0, 0, 0, 1, 1, 1, 1]
  Label alignment  : normal
  Accuracy         : 75.0%
  doc1 (Sports ) → cluster 1  ✗
  doc2 (Sports ) → cluster 0  ✓
  doc3 (Sports ) → cluster 0  ✓
  doc4 (Sports ) → cluster 0  ✓
  doc5 (Finance) → cluster 1  ✓
  doc6 (Finance) → cluster 0  ✗
  doc7 (Finance) → cluster 1  ✓
  doc8 (Finance) → cluster 1  ✓

  Results for: sklearn KMeans — 1-gram
  Predicted labels : [np.int32(1), np.int32(1), np.int32(0), np.int32(0), np

In [53]:

sports_docs = [doc1, doc2, doc3, doc4]
finance_docs = [doc5, doc6, doc7, doc8]

def get_token_set(docs, n=1):
    all_ng = set()
    for doc in docs:
        tokens = preprocess_text(doc)
        for ng in extract_ngrams(tokens, n):
            all_ng.add(ng)
    return all_ng


print("=== 1-gram Analysis ===")
sports_1 = get_token_set(sports_docs, 1)
finance_1 = get_token_set(finance_docs, 1)
shared_1 = sports_1 & finance_1
only_sports_1 = sports_1 - finance_1
only_finance_1 = finance_1 - sports_1

print(f"  Shared tokens (hard to separate): {sorted(shared_1)}")
print(f"  Only in Sports  : {sorted(only_sports_1)}")
print(f"  Only in Finance : {sorted(only_finance_1)}")

print()
print("=== 2-gram Analysis ===")
sports_2 = get_token_set(sports_docs, 2)
finance_2 = get_token_set(finance_docs, 2)
shared_2 = sports_2 & finance_2
only_sports_2 = sports_2 - finance_2
only_finance_2 = finance_2 - sports_2

print(f"  Shared bigrams  (hard to separate): {sorted(shared_2)}")
print(f"  Only in Sports  : {sorted(only_sports_2)}")
print(f"  Only in Finance : {sorted(only_finance_2)}")

print()
rati_1 = len(shared_1) / (len(sports_1 | finance_1)) * 100
ratio_2 = len(shared_2) / (len(sports_2 | finance_2)) * 100
print(f"  Overlap ratio — 1-gram: {rati_1:.1f}%")
print(f"  Overlap ratio — 2-gram: {ratio_2:.1f}%")
print()
print("→ Lower overlap means the vectors are MORE distinct → easier for KMeans.")

=== 1-gram Analysis ===
  Shared tokens (hard to separate): ['a', 'all', 'for', 'gold', 'high', 'is', 'market', 'needs', 'of', 'price', 'the', 'trade', 'will']
  Only in Sports  : ['athlete', 'effort', 'jump', 'medal', 'sweat', 'winning']
  Only in Finance : ['bank', 'bars', 'in', 'investing', 'money', 'rate', 'today']

=== 2-gram Analysis ===
  Shared bigrams  (hard to separate): ['a high', 'a trade', 'all for', 'is a', 'is high', 'market for', 'needs a', 'price is', 'the gold', 'trade all', 'trade of', 'will trade']
  Only in Sports  : ['a gold', 'athlete will', 'for a', 'gold medal', 'high effort', 'high jump', 'medal is', 'medal needs', 'medal price', 'of sweat', 'the athlete', 'winning a']
  Only in Finance : ['bank will', 'bars is', 'bars needs', 'bars price', 'for gold', 'gold bars', 'high rate', 'high today', 'in gold', 'investing in', 'of money', 'the bank']

  Overlap ratio — 1-gram: 50.0%
  Overlap ratio — 2-gram: 33.3%

→ Lower overlap means the vectors are MORE distinct → 

In [54]:
D1 = "I love cats"
D2 = "Cats are chill"
D3 = "I am late"


def add_padding(tokens, window_size=1):
    padded = ["<s>"] * window_size + tokens + ["</s>"] * window_size
    return padded


def extract_windows(tokens, window_size=1):
    total_width = 2 * window_size + 1
    padded = add_padding(tokens, window_size)

    windows = []
    for i in range(len(padded) - total_width + 1):
        window = padded[i : i + total_width]
        windows.append(" ".join(window))

    return windows


def build_vocab_windows(all_docs_windows):
    all_windows = set()
    for doc_windows in all_docs_windows:
        for w in doc_windows:
            all_windows.add(w)

    vocab = sorted(list(all_windows))
    vocab_index = {w: i for i, w in enumerate(vocab)}
    return vocab, vocab_index


def vectorize_context_window(doc_windows, vocab_index):
    doc_set = set(doc_windows)
    vector = [0] * len(vocab_index)
    for window, idx in vocab_index.items():
        if window in doc_set:
            vector[idx] = 1
    return vector


docs_task2 = [D1, D2, D3]

all_windows_per_doc = []
for doc in docs_task2:
    tokens = preprocess_text(doc)
    wins = extract_windows(tokens, window_size=1)
    all_windows_per_doc.append(wins)
    print(f"Doc: '{doc}'")
    print(f"  Tokens: {tokens}")
    print(f"  Windows: {wins}")
    print()

Doc: 'I love cats'
  Tokens: ['i', 'love', 'cats']
  Windows: ['<s> i love', 'i love cats', 'love cats </s>']

Doc: 'Cats are chill'
  Tokens: ['cats', 'are', 'chill']
  Windows: ['<s> cats are', 'cats are chill', 'are chill </s>']

Doc: 'I am late'
  Tokens: ['i', 'am', 'late']
  Windows: ['<s> i am', 'i am late', 'am late </s>']



In [55]:
vocab_w, vocab_idx_w = build_vocab_windows(all_windows_per_doc)

print("=== Context Window Vocabulary (sorted) ===")
for i, v in enumerate(vocab_w):
    print(f"  [{i}] {v}")

print(f"\nVocab size: {len(vocab_w)}")

=== Context Window Vocabulary (sorted) ===
  [0] <s> cats are
  [1] <s> i am
  [2] <s> i love
  [3] am late </s>
  [4] are chill </s>
  [5] cats are chill
  [6] i am late
  [7] i love cats
  [8] love cats </s>

Vocab size: 9


In [56]:

print("\n=== Context Window BoW Vectors ===")
context_vectors = []
for i, (doc, wins) in enumerate(zip(docs_task2, all_windows_per_doc)):
    vec = vectorize_context_window(wins, vocab_idx_w)
    context_vectors.append(vec)
    print(f"D{i+1} ('{doc}'): {vec}")

print()

print("=" * 60)
print("Full matrix (rows=docs, cols=windows):")
print()

col_labels = [f"[{i}]" for i in range(len(vocab_w))]
print("      " + " ".join(col_labels))
print("      " + "-" * (4 * len(vocab_w)))

for i, vec in enumerate(context_vectors):
    row = " ".join([f"  {v} " for v in vec])
    print(f"  D{i+1}: {row}")

print()
print("Window index reference:")
for i, v in enumerate(vocab_w):
    print(f"  [{i}] = '{v}'")


=== Context Window BoW Vectors ===
D1 ('I love cats'): [0, 0, 1, 0, 0, 0, 0, 1, 1]
D2 ('Cats are chill'): [1, 0, 0, 0, 1, 1, 0, 0, 0]
D3 ('I am late'): [0, 1, 0, 1, 0, 0, 1, 0, 0]

Full matrix (rows=docs, cols=windows):

      [0] [1] [2] [3] [4] [5] [6] [7] [8]
      ------------------------------------
  D1:   0    0    1    0    0    0    0    1    1 
  D2:   1    0    0    0    1    1    0    0    0 
  D3:   0    1    0    1    0    0    1    0    0 

Window index reference:
  [0] = '<s> cats are'
  [1] = '<s> i am'
  [2] = '<s> i love'
  [3] = 'am late </s>'
  [4] = 'are chill </s>'
  [5] = 'cats are chill'
  [6] = 'i am late'
  [7] = 'i love cats'
  [8] = 'love cats </s>'


The TP gives us the expected vocabulary for D1, D2, D3 with context window size=1:

```
0: "<s> cats are"
1: "<s> i am"
2: "<s> i love"
3: "am late </s>"
4: "are chill </s>"
5: "cats are chill"
6: "i am late"
7: "i love cats"
8: "love cats </s>"
```

Let me verify our output matches.

In [57]:
expected_vocab = [
    "<s> cats are",
    "<s> i am",
    "<s> i love",
    "am late </s>",
    "are chill </s>",
    "cats are chill",
    "i am late",
    "i love cats",
    "love cats </s>",
]

print("=== Vocabulary Verification ===")
print(f"Expected size : {len(expected_vocab)}")
print(f"Our size      : {len(vocab_w)}")
print()

all_match = True
for i, (exp, got) in enumerate(zip(expected_vocab, vocab_w)):
    match = "✓" if exp == got else "✗"
    if exp != got:
        all_match = False
    print(f"  [{i}] {match}  Expected: '{exp}'  |  Got: '{got}'")

print()
if all_match:
    print("✓ Our vocabulary matches the expected TP output perfectly!")
else:
    print("✗ There's a mismatch somewhere — check preprocessing steps.")

=== Vocabulary Verification ===
Expected size : 9
Our size      : 9

  [0] ✓  Expected: '<s> cats are'  |  Got: '<s> cats are'
  [1] ✓  Expected: '<s> i am'  |  Got: '<s> i am'
  [2] ✓  Expected: '<s> i love'  |  Got: '<s> i love'
  [3] ✓  Expected: 'am late </s>'  |  Got: 'am late </s>'
  [4] ✓  Expected: 'are chill </s>'  |  Got: 'are chill </s>'
  [5] ✓  Expected: 'cats are chill'  |  Got: 'cats are chill'
  [6] ✓  Expected: 'i am late'  |  Got: 'i am late'
  [7] ✓  Expected: 'i love cats'  |  Got: 'i love cats'
  [8] ✓  Expected: 'love cats </s>'  |  Got: 'love cats </s>'

✓ Our vocabulary matches the expected TP output perfectly!
